# Retail Sales Forecasting & Inventory Optimization System
### End-to-End Analytics, Machine Learning & Supply Chain Simulation Walkthrough

**Author:** Retail Analytics & Data Science Team  
**Scope:** Multi-Store Daily Retail Demand Forecasting & Multi-Echelon Replenishment Planning

---

## 1. Environment Setup and Library Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generator import generate_synthetic_retail_data
from src.data_preprocessing import run_preprocessing_pipeline
from src.eda import calculate_abc_xyz_classification, generate_eda_figures
from src.feature_engineering import build_feature_pipeline
from src.forecasting import chronological_split, train_and_evaluate_forecaster, plot_forecast_vs_actuals
from src.inventory_optimization import optimize_inventory_portfolio, load_inventory_config
from src.business_insights import generate_executive_report

print("All project modules imported successfully!")

## 2. Ingest or Generate Retail Sales Data
Generates 2 years of daily retail sales across 2 stores and 10 SKUs featuring trends, weekend seasonality, promotional lifts, and customer returns.

In [ ]:
df_raw = generate_synthetic_retail_data(output_path="../data/raw/sales_data.csv")
df_raw.head()

## 3. Data Preprocessing & Validation
Validates schema, separates customer return transactions, aggregates to daily store-SKU grain, and builds an unbroken continuous calendar grid.

In [ ]:
df_clean = run_preprocessing_pipeline(
    raw_data_path="../data/raw/sales_data.csv",
    output_path="../data/processed/clean_sales_data.csv"
)
df_clean.info()

## 4. Exploratory Data Analysis & Pareto ABC-XYZ Classification
- **ABC Analysis:** Revenue contribution (80% / 15% / 5% Pareto distribution)
- **XYZ Analysis:** Demand volatility ($CV = \sigma / \mu$)

In [ ]:
sku_stats = calculate_abc_xyz_classification(df_clean)
generate_eda_figures(df_clean, sku_stats, output_dir="../reports/figures")
sku_stats[["sku_id", "category", "total_revenue", "cv_demand", "abc_class", "xyz_class", "segment"]]

## 5. Feature Engineering & Strict Chronological Splitting
- Builds trigonometric cyclical encodings (sin/cos of Day-of-Week and Month)
- Calculates strictly lagged historical features ($lag_1, lag_7, lag_{14}, lag_{28}$)
- Computes rolling averages on `shift(1)` data to eliminate lookahead data leakage
- Partitions into Train, Validation, and Holdout Test sets

In [ ]:
df_feat, feature_cols = build_feature_pipeline(df_clean)
train_df, val_df, test_df = chronological_split(df_feat, test_days=28, val_days=28)

print(f"Feature matrix shape: {df_feat.shape}")
print(f"Feature columns count: {len(feature_cols)}")

## 6. Model Training & Heuristic Benchmark Comparison
Fits a Random Forest Regressor and evaluates against Naive, 7-Day Moving Average, and Seasonal Naive baselines.

In [ ]:
model, metrics_df, test_results = train_and_evaluate_forecaster(
    train_df, val_df, test_df, feature_cols,
    model_output_path="../models/random_forest_forecaster.joblib"
)
metrics_df

In [ ]:
plot_forecast_vs_actuals(test_results, output_fig_path="../reports/figures/forecast_vs_actuals.png")

# Display plot inline
from IPython.display import Image, display
display(Image(filename="../reports/figures/forecast_vs_actuals.png"))

## 7. Inventory Optimization & Replenishment Planning
Calculates:
- Safety Stock ($SS = Z \times \sqrt{L} \times \sigma_d$)
- Reorder Point ($ROP = d_L + SS$)
- Economic Order Quantity ($EOQ = \sqrt{2DS/H}$)
- Discrete packaging constraints (MOQ and Case Pack rounding)

In [ ]:
config = load_inventory_config("../config/inventory_config.yaml")
inventory_plan = optimize_inventory_portfolio(test_results, config)

summary_cols = [
    "store_id", "sku_id", "category", "daily_forecast", "safety_stock", 
    "reorder_point", "current_on_hand", "days_of_supply", "recommended_order_qty", 
    "order_value_usd", "health_status"
]
inventory_plan[summary_cols]

## 8. Business Insights & Executive Reporting

In [ ]:
report = generate_executive_report(
    inventory_plan_path="../reports/inventory_replenishment_orders.csv",
    metrics_path="../reports/forecast_metrics.csv",
    classification_path="../reports/sku_abc_xyz_classification.csv",
    output_report_path="../reports/executive_inventory_summary.md"
)
print("Executive summary brief generated successfully!")